# Pruebas $F$: hipótesis conjuntas

*¿Cómo evaluamos varias afirmaciones sobre un modelo al mismo tiempo?*

En el notebook 08 utilizamos la prueba $t$ para contrastar hipótesis sobre un parámetro. Ahora examinaremos **restricciones lineales conjuntas**. Primero construiremos la prueba $F$ comparando modelos restringidos y no restringidos; después recuperaremos su expresión matricial y comprobaremos los resultados con R.

Continuaremos con el caso de Big Andy's Burger Barn. Este notebook puede ejecutarse de forma independiente.

## Importamos los datos

Usamos nuevamente la base de Hill, Griffiths y Lim (2011), con 75 observaciones de franquicias. `sales` representa ventas mensuales (miles de dólares), `advert` gasto mensual en publicidad (miles de dólares) y `price` un índice de precios expresado en dólares.

La ruta es la misma de los notebooks 06, 07 y 08.

In [ ]:
library(readr)

datos <- read_csv("../data/andy.csv")
head(datos)

## Ampliamos el modelo

En los notebooks anteriores estimamos una relación lineal entre ventas, precio y publicidad. Ahora permitimos que la relación entre publicidad y ventas sea cuadrática:

$$
sales_i=\beta_1+\beta_2price_i+\beta_3advert_i+\beta_4advert_i^2+u_i.
$$

El modelo sigue siendo **lineal en los parámetros**. El efecto marginal de la publicidad depende de su nivel:

$$
\frac{\partial E(sales_i)}{\partial advert_i}=\beta_3+2\beta_4advert_i.
$$

Usamos esta especificación para formular preguntas sobre varios parámetros, no para demostrar que existe una relación causal.

In [ ]:
mco <- lm(sales ~ price + advert + I(advert^2), data = datos)
coef(mco)

## Recuperamos la variabilidad residual

El **modelo no restringido** permite estimar libremente los cuatro coeficientes. Necesitaremos su suma de cuadrados residual y los grados de libertad:

$$
RSS_U=\sum_{i=1}^{n}e_{Ui}^2,
\qquad
s_U^2=\frac{RSS_U}{n-k}.
$$

La función `deviance()` devuelve $RSS$ para un modelo estimado con `lm()`. Aquí $k$ incluye el intercepto.

In [ ]:
n <- nobs(mco)
k <- length(coef(mco))
gl <- df.residual(mco)
RSS_U <- deviance(mco)
s2_U <- RSS_U / gl

c(n = n, k = k, gl = gl, RSS_U = RSS_U, s2_U = s2_U)

## ¿Es significativo el modelo en su conjunto?

Comenzamos con la **hipótesis de significancia global**:

$$
H_0:\beta_2=\beta_3=\beta_4=0.
$$

Si estas tres restricciones fueran ciertas, las ventas esperadas serían constantes: solamente necesitaríamos un intercepto. Ese es nuestro **modelo restringido**.

El contraste no pregunta si las tres estimaciones son distintas de cero *por separado*: pregunta si los datos son compatibles con que las tres restricciones se cumplan simultáneamente.

In [ ]:
mod_constante <- lm(sales ~ 1, data = datos)
RSS_R_global <- deviance(mod_constante)
m_global <- 3

c(RSS_restringido = RSS_R_global, RSS_no_restringido = RSS_U,
  aumento_RSS = RSS_R_global - RSS_U)

## Construimos el estadístico $F$ mediante $RSS$

Imponer restricciones no puede reducir el $RSS$ mínimo: $RSS_R\geq RSS_U$. Medimos cuánto empeora el ajuste, en relación con la variabilidad residual del modelo no restringido:

$$
F=\frac{(RSS_R-RSS_U)/m}{RSS_U/(n-k)}.
$$

Aquí $m=3$. Bajo $H_0$, los supuestos clásicos y la normalidad de los errores, el estadístico tiene distribución $F_{m,n-k}$. Para la inferencia exacta seguimos tratando los regresores como deterministas.

In [ ]:
F_global <- ((RSS_R_global - RSS_U) / m_global) / s2_U
F_global

## Valores críticos, valor $p$ y decisión

La prueba $F$ rechaza en la **cola derecha**: valores grandes indican que las restricciones empeoran mucho el ajuste. Para cada nivel de significancia $\alpha$ calculamos el valor crítico con `qf()` y el valor $p$ con `pf()`.

$$
p=P(F_{m,n-k}\geq F_{obs}\mid H_0).
$$

Rechazamos $H_0$ cuando $F_{obs}>F_{\alpha;m,n-k}$, o equivalentemente cuando $p<\alpha$.

In [ ]:
alpha <- c(0.10, 0.05, 0.01)
F_critico <- qf(1 - alpha, df1 = m_global, df2 = gl)
p_global <- pf(F_global, df1 = m_global, df2 = gl, lower.tail = FALSE)

data.frame(alpha, F_observado = F_global, F_critico,
           valor_p = p_global,
           decision = ifelse(p_global < alpha, "Rechazar H0", "No rechazar H0"))

## Visualizamos la regla de decisión

La distribución $F$ no tiene valores negativos. La línea discontinua marca el valor crítico al 5%; la región sombreada, a su derecha, es la **región de rechazo**.

En estos datos el estadístico observado es mucho mayor que el valor crítico. Para apreciar la región relevante de la distribución, mostramos solamente su parte inicial y anotamos el estadístico, que queda fuera de la escala de la gráfica.

In [ ]:
Fc05 <- qf(0.95, df1 = m_global, df2 = gl)
limite <- qf(0.995, df1 = m_global, df2 = gl)

curve(df(x, df1 = m_global, df2 = gl), from = 0, to = limite,
      xlab = "Estadístico F", ylab = "Densidad", main = "")
cola <- seq(Fc05, limite, length.out = 100)
polygon(c(Fc05, cola, limite), c(0, df(cola, m_global, gl), 0),
        col = "gray90", border = NA)
abline(v = Fc05, lty = 2)
mtext(sprintf("F observado = %.2f (fuera de escala)", F_global),
      side = 3, line = 0.3, cex = 0.85)


## Verificamos la significancia global

`anova()` puede comparar dos **modelos anidados** estimados sobre las mismas observaciones. La prueba corresponde a la pérdida de ajuste al pasar del modelo no restringido al restringido.

Podemos comprobar además que el estadístico coincide con el `F-statistic` del resumen del modelo completo. También la calculamos con $R^2$ del modelo completo:

$$
F=\frac{R_U^2/(k-1)}{(1-R_U^2)/(n-k)}.
$$

Esta última expresión corresponde a la prueba de significancia global con constante.

In [ ]:
print(anova(mod_constante, mco))

R2_U <- summary(mco)$r.squared
F_R2 <- (R2_U / (k - 1)) / ((1 - R2_U) / gl)

print(c(F_RSS = F_global, F_R2 = F_R2,
        F_resumen = unname(summary(mco)$fstatistic["value"])))

## ¿Aporta información la publicidad en su conjunto?

La publicidad aparece en dos términos del modelo. Para preguntar si **ambos términos pueden excluirse simultáneamente**, contrastamos:

$$
H_0:\beta_3=\beta_4=0.
$$

El modelo restringido conserva `price`, pero excluye `advert` y `advert` al cuadrado. Ahora tenemos $m=2$ restricciones.

In [ ]:
mod_precio <- lm(sales ~ price, data = datos)
RSS_R_publicidad <- deviance(mod_precio)
m_publicidad <- 2

F_publicidad <- ((RSS_R_publicidad - RSS_U) / m_publicidad) / s2_U
p_publicidad <- pf(F_publicidad, m_publicidad, gl, lower.tail = FALSE)

c(RSS_R = RSS_R_publicidad, RSS_U = RSS_U,
  F_observado = F_publicidad, valor_p = p_publicidad)

## Comprobamos el contraste de publicidad

En esta comparación ambos modelos utilizan la misma variable dependiente y las mismas observaciones. `anova()` debe reproducir el estadístico calculado con las sumas de cuadrados residuales.

Una prueba $t$ sobre `advert` o sobre `I(advert^2)` considera **una restricción a la vez**. No reemplaza la prueba conjunta, que también depende de la covarianza entre los estimadores.

In [ ]:
print(anova(mod_precio, mco))
print(summary(mco)$coefficients)

## Representamos las restricciones con matrices

La misma hipótesis $H_0:\beta_3=\beta_4=0$ puede escribirse como

$$
H_0:\mathbf R\boldsymbol\beta=\mathbf r,
\qquad
\mathbf R=\begin{pmatrix}0&0&1&0\\0&0&0&1\end{pmatrix},
\qquad
\mathbf r=\begin{pmatrix}0\\0\end{pmatrix}.
$$

Con el orden de coeficientes $(\beta_1,\beta_2,\beta_3,\beta_4)$, cada renglón de $\mathbf R$ establece una restricción. Como los dos renglones son independientes, $m=2$.

In [ ]:
b <- coef(mco)
Vb <- vcov(mco)

R_pub <- rbind(c(0, 0, 1, 0), c(0, 0, 0, 1))
colnames(R_pub) <- names(b)
r_pub <- c(0, 0)

print(R_pub)
print(r_pub)

## Calculamos $F$ con la matriz de varianzas y covarianzas

La expresión matricial desarrollada en clase es

$$
F=\frac{(\mathbf R\mathbf b-\mathbf r)^\top
[\mathbf R(\mathbf X^\top\mathbf X)^{-1}\mathbf R^\top]^{-1}
(\mathbf R\mathbf b-\mathbf r)}{m\,s_U^2}.
$$

Como `vcov(mco)` devuelve $\mathbf V_b=s_U^2(\mathbf X^\top\mathbf X)^{-1}$, podemos escribir de forma equivalente

$$
F=\frac{(\mathbf R\mathbf b-\mathbf r)^\top
(\mathbf R\mathbf V_b\mathbf R^\top)^{-1}
(\mathbf R\mathbf b-\mathbf r)}{m}.
$$

Así recuperamos la misma prueba sin estimar explícitamente un segundo modelo.

In [ ]:
d_pub <- R_pub %*% b - r_pub
V_pub <- R_pub %*% Vb %*% t(R_pub)

F_matriz_pub <- as.numeric(t(d_pub) %*% solve(V_pub) %*% d_pub) / nrow(R_pub)

print(c(F_RSS = F_publicidad, F_matriz = F_matriz_pub))
print(all.equal(unname(F_publicidad), unname(F_matriz_pub)))

## Una restricción económicamente interpretable

En el modelo cuadrático, el efecto marginal de la publicidad es $\beta_3+2\beta_4advert$. Si la publicidad vale $1.8$ (es decir, 1,800 dólares al mes), una hipótesis posible es:

$$
H_0:\beta_3+3.6\beta_4=1.
$$

La hipótesis plantea que, en ese nivel de publicidad, **el efecto marginal** sobre las ventas mensuales esperadas es de mil dólares por cada mil dólares de publicidad. Se trata de una tasa de cambio *local*, no necesariamente del efecto de aumentar discretamente el gasto publicitario en mil dólares.

Es **una sola restricción lineal**, aunque incluya dos parámetros.

In [ ]:
L <- c(0, 0, 1, 3.6)
names(L) <- names(b)

efecto_estimado <- sum(L * b)
se_efecto <- sqrt(as.numeric(t(L) %*% Vb %*% L))
t_efecto <- (efecto_estimado - 1) / se_efecto
F_efecto <- t_efecto^2
p_efecto <- pf(F_efecto, 1, gl, lower.tail = FALSE)

c(efecto_estimado = efecto_estimado, error_estandar = se_efecto,
  t_observado = t_efecto, F_observado = F_efecto, valor_p = p_efecto)

## Una sola restricción: $F=t^2$

Cuando $m=1$, la prueba $F$ de una restricción lineal y la prueba $t$ bilateral (de dos colas) de esa misma restricción son equivalentes:

$$F=t^2.$$

El cuadrado elimina el signo del estadístico $t$, de modo que la prueba $F$ no distingue una alternativa de cola izquierda de una de cola derecha. También podemos comprobar la equivalencia de sus valores $p$ bilaterales.

In [ ]:
p_t_bilateral <- 2 * pt(abs(t_efecto), gl, lower.tail = FALSE)

print(c(F = F_efecto, t_cuadrada = t_efecto^2,
        valor_p_F = p_efecto, valor_p_t_bilateral = p_t_bilateral))

## Imponemos la restricción en un modelo

También podemos calcular la prueba anterior usando $RSS$. La hipótesis $\beta_3+3.6\beta_4=1$ implica $\beta_3=1-3.6\beta_4$. Al sustituir en la ecuación original, obtenemos:

$$
sales_i-advert_i=\beta_1+\beta_2price_i
+\beta_4(advert_i^2-3.6\,advert_i)+u_i.
$$

Estimaremos este **modelo restringido**. Usamos `I()` para que R interprete las restas y potencias como operaciones aritméticas dentro de la fórmula.

In [ ]:
mod_efecto_restringido <- lm(
  I(sales - advert) ~ price + I(advert^2 - 3.6 * advert),
  data = datos
)

RSS_R_efecto <- deviance(mod_efecto_restringido)
F_RSS_efecto <- (RSS_R_efecto - RSS_U) / s2_U

print(c(RSS_R = RSS_R_efecto, RSS_U = RSS_U,
        F_RSS = F_RSS_efecto, F_t_cuadrada = F_efecto))
print(all.equal(unname(F_RSS_efecto), unname(F_efecto)))

## Precaución con la $R^2$ del modelo transformado

Los dos modelos anteriores generan residuos comparables y, por tanto, podemos usar directamente $RSS_R$ y $RSS_U$. Sin embargo, la variable dependiente del restringido es `sales - advert`, no `sales`.

**No debemos sustituir sin más sus $R^2$ automáticas** en la fórmula de $F$ basada en $R^2$: las sumas de cuadrados totales podrían ser diferentes. La fórmula basada en $R^2$ sí es válida para los modelos anidados anteriores que comparten variable dependiente y muestra.

## Planteamos dos afirmaciones simultáneas

Además de la hipótesis sobre el efecto marginal de publicidad, queremos evaluar que las ventas mensuales esperadas sean 80 mil dólares cuando `price = 6` y `advert = 1.8`:

$$
\begin{aligned}
H_0:\quad \beta_3+3.6\beta_4 &= 1,\\
\beta_1+6\beta_2+1.8\beta_3+3.24\beta_4 &= 80.
\end{aligned}
$$

La segunda ecuación corresponde a $E(sales)$ en esos valores de los regresores. **La prueba conjunta exige que se cumplan ambas afirmaciones**, no solo una.

In [ ]:
R_conj <- rbind(c(0, 0, 1, 3.6),
               c(1, 6, 1.8, 3.24))
colnames(R_conj) <- names(b)
r_conj <- c(1, 80)

print(R_conj)
print(r_conj)

## Calculamos la prueba conjunta

Ahora $m=2$. Volvemos a utilizar el estadístico matricial y la distribución $F_{2,n-k}$. La prueba considera tanto las discrepancias de cada restricción como sus covarianzas; no es correcto combinar los valores $p$ de dos pruebas separadas.

In [ ]:
d_conj <- R_conj %*% b - r_conj
V_conj <- R_conj %*% Vb %*% t(R_conj)

F_conj <- as.numeric(t(d_conj) %*% solve(V_conj) %*% d_conj) / nrow(R_conj)
p_conj <- pf(F_conj, nrow(R_conj), gl, lower.tail = FALSE)

print(c(F_observado = F_conj, valor_p = p_conj))
print(ifelse(p_conj < 0.05, "Rechazar H0 al 5%", "No rechazar H0 al 5%"))

## Comprobamos los contrastes con `car`

`car::linearHypothesis()` permite contrastar restricciones escritas en forma matricial. La utilizamos **solo para verificar** los cálculos anteriores. El paquete `car` es opcional para ejecutar el resto del notebook; si no está instalado, el trabajo manual permanece completo.

In [ ]:
if (requireNamespace("car", quietly = TRUE)) {
  print(car::linearHypothesis(mco, R_pub, rhs = r_pub))
  print(car::linearHypothesis(mco, R_conj, rhs = r_conj))
} else {
  cat("Verificación opcional: instala el paquete 'car' para ejecutar linearHypothesis().\n")
}

## Interpretación de la evidencia

En el modelo cuadrático de Andy obtenemos resultados distintos porque cada prueba responde una pregunta diferente (valores aproximados):

- **Significancia global** ($F\approx24.46$; $p<0.001$): rechazamos que los tres coeficientes de pendiente sean simultáneamente cero. Esto indica que, al menos, uno difiere de cero bajo los supuestos del modelo.
- **Publicidad conjunta** ($F\approx8.44$; $p\approx0.0005$): rechazamos que los términos lineal y cuadrático de publicidad sean ambos cero, manteniendo el precio en el modelo.
- **Efecto marginal de publicidad igual a 1** ($F\approx3.24$; $p\approx0.076$): no rechazamos la hipótesis al 5%. Esto no demuestra que el efecto marginal sea exactamente 1.
- **Dos restricciones simultáneas** ($F\approx8.35$; $p\approx0.0006$): rechazamos que sean compatibles *conjuntamente* con los datos la afirmación sobre el efecto marginal y la afirmación sobre ventas esperadas de 80 mil dólares.

No hay contradicción entre las últimas dos decisiones: una prueba examina una sola restricción y la otra exige que se cumplan ambas. El rechazo conjunto, por sí solo, no identifica cuál restricción explica la discrepancia.

El valor $p$ mide qué tan extremo resulta el estadístico bajo la hipótesis nula y los supuestos del modelo; no es la probabilidad de que $H_0$ sea verdadera. Tampoco una prueba significativa demuestra causalidad o importancia económica.

## Para reflexionar

1. ¿Por qué una prueba $t$ sobre el coeficiente de `advert` no responde la misma pregunta que $H_0:\beta_3=\beta_4=0$?
2. ¿Por qué el modelo restringido nunca tiene un $RSS$ menor que el modelo no restringido, si ambos están estimados sobre la misma muestra y el primero realmente impone restricciones al segundo?
3. ¿Cómo explicas que $F=t^2$ cuando hay una sola restricción, aunque el estadístico $F$ nunca sea negativo?
4. ¿Por qué sería incorrecto usar directamente los dos valores de $R^2$ reportados después de transformar `sales` en `sales - advert`?
5. Si la prueba conjunta rechaza, ¿podemos identificar con ese resultado cuál de las dos restricciones es incompatible con los datos?

## Documentación

- `lm()`, `anova()` y `vcov()` en la documentación de R: [modelos lineales](https://stat.ethz.ch/R-manual/R-patched/library/stats/html/lm.html), [ANOVA](https://stat.ethz.ch/R-manual/R-patched/library/stats/html/anova.lm.html) y [matriz de covarianzas](https://stat.ethz.ch/R-manual/R-patched/library/stats/html/vcov.html).
- `linearHypothesis()` en [car](https://cran.r-project.org/package=car), para verificar restricciones lineales.